In [ ]:
import random, uuid

customers_int = 10
ts_step_int = 1

class OrderGenerator:
    def __init__(self):
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = ts_step_int

    def generate(self):
        order_id_str = str(uuid.uuid4())
        message_dict = {
            "key": order_id_str,
            "value": {"order_id": order_id_str,
                      "customer_id": random.randint(0, customers_int - 1),
                      "price": random.randint(1, 10000) / 100,
                      "ts": self.ts_int},
        }
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
sink_str = "orders_tumbling"
#
size_int = ts_step_int * 100
allowed_lateness_int = size_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire_tumbling(lambda x: x["ts"], size_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_tumbling(size_int,
                                     lambda x: x["ts"],
                                     lambda x: x["customer_id"],
                                     lambda agg, x: {"orders": agg["orders"] + 1,
                                                     "total_price": agg["total_price"] + x["price"]},
                                     {"orders": 0, "total_price": 0},
                                     lambda by, agg: {"customer_id": by,
                                                      "orders": agg["orders"],
                                                      "total_price": agg["total_price"]},
                                     lambda l: {**l[0], "window_end": l[1]},
                                     trigger_positive_only=False)
#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)

In [ ]:
# --- RETRACTION ASSERTION ---

def assert_output(actual, expected_tuples):
    if isinstance(actual, dict) and sink_str in actual:
        actual = actual[sink_str]
        
    actual_processed = []
    for record, w in actual:
        actual_processed.append((record, w))
        
    def sort_key(item):
        d, w = item
        return (d.get("window_end", 0), d.get("customer_id", 0), w)
        
    actual_sorted = sorted(actual_processed, key=sort_key)
    expected_sorted = sorted(expected_tuples, key=sort_key)
    
    if actual_sorted != expected_sorted:
        raise ValueError(f"\nAssertion Failed!\nExpected: {expected_sorted}\nGot:      {actual_sorted}")

def push_with_weight(customer_id, price, ts, weight=1):
    """Pusht Daten über die _from_records Schnittstelle direkt mit Gewicht."""
    record = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    built_tn.push(order_source_str, [(record, weight)])
    return built_tn.latest()

# --- COMPLETE TEST RUN ---
built_tn.reset()

print("=== Step 1: Normaler Aufbau des Fensters ===")
push_with_weight(customer_id=1, price=100, ts=10, weight=1)
push_with_weight(customer_id=1, price=200, ts=50, weight=1)
print("-> OK: Daten im ersten Fenster gesammelt.")

print("\n=== Step 2: Fenster [0,100) schließen durch Watermark ===")
res = push_with_weight(customer_id=2, price=50, ts=105, weight=1)
assert_output(res, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, 1)
])
print("-> OK: Fenster für Kunde 1 emittiert (orders=2, total=300).")

print("\n=== Step 3: Eingehende Retraction (-1) für ein bereits emittiertes Fenster ===")
res = push_with_weight(customer_id=1, price=200, ts=50, weight=-1)
assert_output(res, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, -1),
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, 1)
])
print("-> OK: Reinlaufende Retraction erfolgreich verarbeitet!")

print("\n=== Step 4: Fenster [0,100) komplett leeren (Retraction auf 0 Orders) ===")
res = push_with_weight(customer_id=1, price=100, ts=10, weight=-1)
assert_output(res, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, -1)
])
print("-> OK: Fenster wurde durch Retraction restlos und sauber im Sink gelöscht.")

print("\n=== Step 5: Punktlandung auf der Fenstergrenze (ts=100) ===")
# Dieses Event landet im Fenster [100, 200)
res = push_with_weight(customer_id=3, price=400, ts=100, weight=1)
assert_output(res, [])
print("-> OK: Event landete im Folgefenster [100,200), kein verfrühtes Feuern.")

print("\n=== Step 6: Weiteres Event im Fenster [100,200) ===")
res = push_with_weight(customer_id=3, price=200, ts=150, weight=1)
assert_output(res, [])
print("-> OK: Weiteres Event im zweiten Fenster gesammelt.")

print("\n=== Step 7: Fenster [100,200) kontrolliert schließen ===")
# ts=210 treibt die Watermark über 200. Das schließt das Fenster [100, 200) endgültig.
res = push_with_weight(customer_id=1, price=50, ts=210, weight=1)
assert_output(res, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 2, "total_price": 600, "window_end": 200}, 1)
])
print("-> OK: Fenster [100,200) wurde für alle enthaltenen Keys korrekt emittiert.")

print("\n=== Step 8: Folgefenster schließen (Kein Retention-Verfall wegen allowed_lateness=300) ===")
# ts=310 schiebt die Watermark auf 310.
# 1. Das schließt das Fenster [200, 300) für Kunde 1 (ts=210 -> orders:1, price:50)
# 2. Da allowed_lateness=300 ist, bleibt das Fenster [100, 200) aktiv im Zustand (Verfall erst ab ts=500!).
res = push_with_weight(customer_id=2, price=60, ts=310, weight=1)
assert_output(res, [
    ({"customer_id": 1, "orders": 1, "total_price": 50, "window_end": 300}, 1)
])
print("-> OK: Folgefenster schließt, altes Fenster bleibt dank hoher Lateness geschützt.")

print("\n=== Step 9: Late Arrival innerhalb der erlaubten Lateness ===")
# Die Watermark steht auf 310. Wir senden ein spätes Event für das bereits geschlossene Fenster [100, 200) mit ts=120.
# Da 310 < 200 + 300 (500), ist das Fenster offen für Korrekturen! Kunde 2 wächst von 1 auf 2 Orders.
res = push_with_weight(customer_id=2, price=40, ts=120, weight=1)
assert_output(res, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 200}, -1),
    ({"customer_id": 2, "orders": 2, "total_price": 90, "window_end": 200}, 1)
])
print("-> OK: Late Arrival verarbeitet und geschlossenes Fenster nachträglich korrigiert.")

print("\n=== Step 10: Ultimativer Retention-Verfall (Watermark treiben über Lateness-Grenze) ===")
# Wir jagen die Watermark mit ts=510 nach vorne.
# 1. Das schließt das bisher offene Fenster [300, 400), in dem das Event aus Step 8 (ts=310) lag.
# 2. Das neue Event (ts=510) landet im Fenster [500, 600) und bleibt mangels Watermark >= 600 noch offen.
# 3. Da 510 >= 200 + 300 (500), fliegt das Fenster [100, 200) jetzt endgültig per Retraction raus!
res = push_with_weight(customer_id=1, price=70, ts=510, weight=1)
assert_output(res, [
    # Retention-Retractions für das abgelaufene Fenster [100, 200)
    ({"customer_id": 2, "orders": 2, "total_price": 90, "window_end": 200}, -1),
    ({"customer_id": 3, "orders": 2, "total_price": 600, "window_end": 200}, -1),
    # Das geschlossene Fenster [300, 400) mit dem Event aus Step 8
    ({"customer_id": 2, "orders": 1, "total_price": 60, "window_end": 400}, 1)
])
print("-> OK: Retention hat das Fenster nach 3x Lateness sauber geräumt und das Zwischenfenster geschlossen.")

print("\n🎉 Gratulation! Alle relationalen und zeitbasierten Edge Cases wurden fehlerfrei bestanden.")


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_hopping"
#
size_int = ts_step_int * 100
hop_int = size_int // 2
allowed_lateness_int = size_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire_hopping(lambda x: x["ts"], size_int, hop_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_hopping(size_int,
                             hop_int,
                             lambda x: x["ts"],
                             lambda x: x["customer_id"],
                             lambda agg, x: {"orders": agg["orders"] + 1,
                                                "total_price": agg["total_price"] + x["price"]},
                             {"orders": 0, "total_price": 0},
                             lambda by, agg: {"customer_id": by,
                                              "orders": agg["orders"],
                                              "total_price": agg["total_price"]},
                             lambda l: {**l[0], "window_end": l[1]},
                             trigger_positive_only=False)
                                        
#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)

In [ ]:
# --- RETRACTION ASSERTION ---

def assert_output(actual, expected_tuples):
    if isinstance(actual, dict) and sink_str in actual:
        actual = actual[sink_str]
        
    actual_processed = []
    for record, w in actual:
        actual_processed.append((record, w))
        
    def sort_key(item):
        d, w = item
        return (d.get("window_end", 0), d.get("customer_id", 0), w)
        
    actual_sorted = sorted(actual_processed, key=sort_key)
    expected_sorted = sorted(expected_tuples, key=sort_key)
    
    if actual_sorted != expected_sorted:
        raise ValueError(f"\nAssertion Failed!\nExpected: {expected_sorted}\nGot:      {actual_sorted}")

def push_with_weight(customer_id, price, ts, weight=1):
    record = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    built_tn.push(order_source_str, [(record, weight)])
    return built_tn.latest()

# --- COMPLETE HOPPING TEST RUN ---
built_tn.reset()

print("=== Step 1: Aufbau des Fensters ===")
# ts=10 fällt in: [0, 100)
push_with_weight(customer_id=1, price=100, ts=10, weight=1)
# ts=60 fällt in: [0, 100) und [50, 150)
push_with_weight(customer_id=1, price=200, ts=60, weight=1)
print("-> OK: Daten gesammelt. Watermark=60.")

print("\n=== Step 2: Erstes Fenster-Raster überschreiten ===")
# ts=110 treibt die Watermark auf 110.
# Das schließt das Fenster [0, 100), da 110 >= 100.
# Das Fenster [50, 150) bleibt offen, da 110 < 150.
res = push_with_weight(customer_id=2, price=50, ts=110, weight=1)
assert_output(res, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, 1)
])
print("-> OK: Erstes Hopping-Fenster emittiert.")

print("\n=== Step 3: Retraction für ein Event in mehreren Fenstern ===")
# Wir stornieren das Event ts=60 (weight=-1).
# Da dieses Event in [0, 100) und [50, 150) liegt:
# 1. Für das geschlossene Fenster [0, 100) wird sofort eine Korrektur emittiert.
# 2. Das offene Fenster [50, 150) merkt sich die Retraction intern im Zustand (kein sofortiger Output).
res = push_with_weight(customer_id=1, price=200, ts=60, weight=-1)
assert_output(res, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, -1),
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, 1)
])
print("-> OK: Inkrementelles Storno für das geschlossene Fenster verarbeitet.")

print("\n=== Step 4: Fenster [50, 150) kontrolliert schließen ===")
# ts=160 schiebt die Watermark auf 160 -> Das schließt das Fenster [50, 150).
# Inhalt von [50, 150) vor dem Schließen:
# - Kunde 1: ts=60 (price=200, weight=1) UND die Retraction (price=200, weight=-1) -> Hebt sich auf (0 Orders)!
# - Kunde 2: ts=110 (price=50, weight=1) -> Bleibt übrig!
# Wenn das Event korrekt in mehreren Fenstern lebte, wird hier jetzt NUR Kunde 2 emittiert!
res = push_with_weight(customer_id=3, price=400, ts=160, weight=1)
assert_output(res, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 150}, 1)
])
print("-> OK: Fenster [50, 150) emittiert, Kunde 1 wurde durch das Storno sauber auf 0 reduziert.")

print("\n=== Step 5: Folgefenster schließen ===")
# ts=260 treibt die Watermark auf 260.
# Das schließt alle Fenster, deren Endzeit <= 260 ist.
# Das betrifft [100, 200) (Kunde 2 und 3) UND [150, 250) (enthält ts=160 von Kunde 3).
res = push_with_weight(customer_id=2, price=90, ts=260, weight=1)
assert_output(res, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 1, "total_price": 400, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 1, "total_price": 400, "window_end": 250}, 1)
])
print("-> OK: Fenster [100, 200) und [150, 250) erfolgreich geschlossen.")

print("\n=== Step 6: Ultimativer Retention-Verfall und Folgefenster ===")
# Wir jagen die Watermark mit ts=460 weit nach vorne.
# 1. Das schließt die Fenster, in denen das Event aus Step 5 (ts=260) lag: [200, 300) und [250, 350).
# 2. Die Retention bereinigt das historische Fenster [0, 100) per Retraction.
res = push_with_weight(customer_id=1, price=10, ts=460, weight=1)
assert_output(res, [
    # Retention-Retraction für das abgelaufene Fenster [0, 100)
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, -1),
    
    # Die regulär geschlossenen Fenster für das Event aus Step 5 (ts=260)
    ({"customer_id": 2, "orders": 1, "total_price": 90, "window_end": 300}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 90, "window_end": 350}, 1)
])
print("-> OK: Inkrementeller Verfall und Multi-Fenster-Ausgabe stimmen exakt überein!")

print("\n🎉 Sämtliche Hopping-Window Edge Cases mit 3x allowed_lateness fehlerfrei bestanden!")


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_cumulative"
#
size_int = ts_step_int * 100
advance_int = size_int // 5
allowed_lateness_int = size_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire_cumulative(lambda x: x["ts"], size_int, advance_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_cumulative(size_int,
                                advance_int,
                                lambda x: x["ts"],
                                lambda x: x["customer_id"],
                                lambda agg, x: {"orders": agg["orders"] + 1,
                                                   "total_price": agg["total_price"] + x["price"]},
                                {"orders": 0, "total_price": 0},
                                lambda by, agg: {"customer_id": by,
                                                 "orders": agg["orders"],
                                                 "total_price": agg["total_price"]},
                                lambda l: {**l[0], "window_end": l[1]},
                                trigger_positive_only=False)

#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)


In [ ]:
# --- RETRACTION ASSERTION ---

def assert_output(actual, expected_tuples):
    if isinstance(actual, dict) and sink_str in actual:
        actual = actual[sink_str]
        
    actual_processed = []
    for record, w in actual:
        data = record["value"] if "value" in record else record
        actual_processed.append((data, w))
        
    def sort_key(item):
        d, w = item
        return (d.get("window_end", 0), d.get("customer_id", 0), w)
        
    actual_sorted = sorted(actual_processed, key=sort_key)
    expected_sorted = sorted(expected_tuples, key=sort_key)
    
    if actual_sorted != expected_sorted:
        raise ValueError(f"\nAssertion Failed!\nExpected: {expected_sorted}\nGot:      {actual_sorted}")

def push_with_weight(customer_id, price, ts, weight=1):
    record = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    built_tn.push(order_source_str, [(record, weight)])
    return built_tn.latest()

# --- COMPLETE CUMULATIVE TEST RUN WITH LATENESS ---
built_tn.reset()

print("=== Step 1: Normaler Aufbau und erstes Schließen ===")
# ts=10 landet in Schritt [0, 20)
push_with_weight(customer_id=1, price=100, ts=10, weight=1)
# ts=30 treibt Watermark auf 30 -> Schließt das erste Teilfenster [0, 20)
res = push_with_weight(customer_id=1, price=200, ts=30, weight=1)
assert_output(res, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 20}, 1)
])
print("-> OK: Erstes kumulatives Teilfenster emittiert.")


print("\n=== Step 2: Kaskadierendes Schließen mehrerer Zwischenstufen ===")
# ts=75 treibt die Watermark auf 75.
# Das schließt die offenen Teilfenster [0, 40) und [0, 60) für alle dort akkumulierten Werte.
# In [0, 40) und [0, 60) liegt: ts=10 (Kunde 1: 100) und ts=30 (Kunde 1: 200) -> orders: 2, price: 300
res = push_with_weight(customer_id=2, price=50, ts=75, weight=1)
assert_output(res, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 40}, 1),
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 60}, 1)
])
print("-> OK: Teilfenster 40 und 60 wurden korrekt geschlossen.")


print("\n=== Step 3: Late Arrival in bereits geschlossene Fenster (Dank Lateness geschützt) ===")
# Watermark steht auf 75. Ein verspätetes Event mit ts=15 (Kunde 1) trifft ein.
# ts=15 gehört in ALLER kumulativen Teilfenster dieses Zyklus ([0, 20), [0, 40), [0, 60) etc.).
# Da 75 < window_end + 300, sind die Zustände voll intakt.
# Für die bereits emittierten Fenster (20, 40, 60) bekommen wir Korrektur-Updates (+1 Order, +50 Price).
# Das offene Fenster [0, 80) sammelt es stillschweigend im Zustand mit.
res = push_with_weight(customer_id=1, price=50, ts=15, weight=1)
assert_output(res, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 20}, -1),
    ({"customer_id": 1, "orders": 2, "total_price": 150, "window_end": 20}, 1),
    
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 40}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 40}, 1),
    
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 60}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 60}, 1)
])
print("-> OK: Late Arrival hat alle betroffenen, bereits geschlossenen Epochen aktualisiert.")


print("\n=== Step 4: Ersten Gesamtblock beenden (Kein Retention-Verfall) ===")
# ts=105 schiebt Watermark auf 105.
# 1. Das schließt die verbleibenden Teilfenster des ersten Blocks: [0, 80) und [0, 100).
#    Darin liegen von Kunde 1: ts=10 (100), ts=30 (200), ts=15 (50) -> orders: 3, price: 350.
#    Darin liegt von Kunde 2: ts=75 (50) -> orders: 1, price: 50.
res = push_with_weight(customer_id=1, price=500, ts=105, weight=1)
assert_output(res, [
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 80}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 80}, 1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 100}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 100}, 1)
])
print("-> OK: Gesamter erster Zyklus [0, 100) vollständig und korrekt emittiert.")


print("\n=== Step 5: Ultimativer Retention-Verfall nach Überschreiten der 3x Lateness ===")
# Wir jagen die Watermark mit ts=410 weit nach vorne.
# 1. Alle Teilfenster des ersten Blocks [0, 100) verfallen (da 410 >= window_end + 300) -> Retractions!
# 2. Alle Teilfenster des zweiten Blocks [100, 200) werden geschlossen, da 410 >= 200 -> Emissionen für ts=105!
res = push_with_weight(customer_id=1, price=10, ts=410, weight=1)
assert_output(res, [
    # --- RETENTION-RETRACTIONS (Erster Block verfällt komplett) ---
    ({"customer_id": 1, "orders": 2, "total_price": 150, "window_end": 20}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 40}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 60}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 80}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 80}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 100}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 100}, -1),
    
    # --- EMISSIONEN (Zweiter Block schließt komplett) ---
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 120}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 140}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 160}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 180}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 200}, 1)
])
print("-> OK: Gesamter Retention-Verfall und Folgelauf-Schließungen fehlerfrei durchlaufen!")

print("\n🎉 Sämtliche Cumulative-Window Edge Cases mit Lateness erfolgreich bestanden!")

In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
sink_str = "orders_sliding"
#
size_int = ts_step_int * 10
allowed_lateness_int = size_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire_sliding(lambda x: x["ts"], size_int, allowed_lateness_int)
)
#
window_tn = order_tn.window_sliding(size_int,
                                    lambda x: x["ts"],
                                    lambda x: x["customer_id"],
                                    lambda agg, x: {"orders": agg["orders"] + 1,
                                                    "total_price": agg["total_price"] + x["price"]},
                                    {"orders": 0, "total_price": 0},
                                    lambda by, agg: {"customer_id": by,
                                                     "orders": agg["orders"],
                                                     "total_price": agg["total_price"]},
                                    lambda l: {**l[0], "window_end": l[1]},
                                    trigger_positive_only=False)

#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)


In [ ]:
# --- RETRACTION ASSERTION ---

def assert_output(actual, expected_tuples):
    if isinstance(actual, dict) and sink_str in actual:
        actual = actual[sink_str]
        
    actual_processed = []
    for record, w in actual:
        data = record["value"] if "value" in record else record
        actual_processed.append((data, w))
        
    def sort_key(item):
        d, w = item
        return (d.get("window_end", 0), d.get("customer_id", 0), w)
        
    actual_sorted = sorted(actual_processed, key=sort_key)
    expected_sorted = sorted(expected_tuples, key=sort_key)
    
    if actual_sorted != expected_sorted:
        raise ValueError(f"\nAssertion Failed!\nExpected: {expected_sorted}\nGot:      {actual_sorted}")

def push_with_weight(customer_id, price, ts, weight=1):
    record = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    built_tn.push(order_source_str, [(record, weight)])
    return built_tn.latest()

# --- COMPLETE SLIDING TEST RUN ---
built_tn.reset()

print("=== Step 1: Erstes datengesteuertes Fenster öffnen ===")
# ts=10 öffnet ein Fenster, das bis 20 geht (10 + size)
res = push_with_weight(customer_id=1, price=100, ts=10, weight=1)
assert_output(res, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 20}, 1)
])
print("-> OK: Erstes datengesteuertes Fenster emittiert.")


print("\n=== Step 2: Watermark erhöhen und Fenster schließen ===")
# ts=25 treibt die Watermark auf 25 -> Schließt das Fenster end=20.
# ts=25 öffnet ein neues Fenster bis 35.
res = push_with_weight(customer_id=2, price=200, ts=25, weight=1)
assert_output(res, [
    ({"customer_id": 2, "orders": 1, "total_price": 200, "window_end": 35}, 1)
])
print("-> OK: Neues Fenster geöffnet, Watermark vorgerückt.")


print("\n=== Step 3: Late Arrival in geschlossenes Fenster ===")
# Watermark steht auf 25. Spätes Event mit ts=15 trifft ein.
# Die Engine zieht die alte Repräsentation bei end=20 ab (-1) 
# und gibt den kombinierten Stand (ts=10 + ts=15) beim neuen gleitenden Ende 25 aus (+1).
res = push_with_weight(customer_id=1, price=50, ts=15, weight=1)
assert_output(res, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 20}, -1),
    ({"customer_id": 1, "orders": 2, "total_price": 150, "window_end": 25}, 1)
])
print("-> OK: Gleitendes Fenster erfolgreich per Late Arrival aktualisiert und verschoben.")


print("\n=== Step 4: Ultimativer Retention-Verfall ===")
# Wir jagen die Watermark mit ts=60 nach vorne.
# 1. Das Fenster end=25 verfällt wegen Lateness (25 + 30 = 55 <= 60) -> Retraction!
# 2. Das neue Event ts=60 (Kunde 1, price=10) öffnet ein Fenster bis 70 und emittiert sofort.
res = push_with_weight(customer_id=1, price=10, ts=60, weight=1)
assert_output(res, [
    # Retention-Retraction für das abgelaufene Fenster von Kunde 1
    ({"customer_id": 1, "orders": 2, "total_price": 150, "window_end": 25}, -1),
    # Sofortige Emission des neuen Fensters für ts=60
    ({"customer_id": 1, "orders": 1, "total_price": 10, "window_end": 70}, 1)
])
print("-> OK: Retention-Verfall und direktes Sliding-Emission stimmen perfekt überein!")

print("\n🎉 Sämtliche Sliding-Window Edge Cases mit Lateness erfolgreich bestanden!")

In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_session"
#
ts_step_int = 1
gap_int = ts_step_int * 20
max_session_int = ts_step_int * 200
allowed_lateness_int = gap_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire_session(lambda x: x["ts"], max_session_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_session(gap_int,
                             lambda x: x["ts"], 
                             lambda x: x["customer_id"], 
                             lambda agg, x: {"orders": agg["orders"] + 1,
                                                "total_price": agg["total_price"] + x["price"]},
                             {"orders": 0, "total_price": 0},
                             lambda by, agg: {"customer_id": by,
                                              "orders": agg["orders"],
                                              "total_price": agg["total_price"]},
                             lambda l: {**l[0], "window_end": l[1]},
                             trigger_positive_only=False)

#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)


In [ ]:
# --- RETRACTION ASSERTION ---

def assert_output(actual, expected_tuples):
    if isinstance(actual, dict) and sink_str in actual:
        actual = actual[sink_str]
        
    actual_processed = []
    for record, w in actual:
        data = record["value"] if "value" in record else record
        actual_processed.append((data, w))
        
    def sort_key(item):
        d, w = item
        return (d.get("window_end", 0), d.get("customer_id", 0), w)
        
    actual_sorted = sorted(actual_processed, key=sort_key)
    expected_sorted = sorted(expected_tuples, key=sort_key)
    
    if actual_sorted != expected_sorted:
        raise ValueError(f"\nAssertion Failed!\nExpected: {expected_sorted}\nGot:      {actual_sorted}")

def push_with_weight(customer_id, price, ts, weight=1):
    record = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    built_tn.push(order_source_str, [(record, weight)])
    return built_tn.latest()

# --- COMPLETE SESSION TEST RUN WITH LATENESS ---
built_tn.reset()

print("=== Step 1: Session aufbauen ===")
push_with_weight(customer_id=1, price=100, ts=10, weight=1)
push_with_weight(customer_id=1, price=150, ts=25, weight=1)  # Delta zur 10 ist 15 (< gap=20)
print("-> OK: Events in aktiver Session gesammelt.")


print("\n=== Step 2: Session schließen via Watermark ===")
# ts=50 treibt Watermark auf 50. 
# Das schließt die erste Session für Kunde 1 exakt bei window_end = 40.
res = push_with_weight(customer_id=2, price=300, ts=50, weight=1)
assert_output(res, [
    ({"customer_id": 1, "orders": 2, "total_price": 250, "window_end": 40}, 1)
])
print("-> OK: Erste Session (end=40) erfolgreich emittiert.")

print("\n=== Step 3: Late Arrival innerhalb der Lateness ===")
# Watermark steht auf 50. Spätes Event mit ts=35 trifft ein.
# Die Engine rechnet das Event in die bestehende Session [10, 40) ein.
# Sie zieht den alten Stand ab (-1) und gibt den neuen Stand sofort aus (+1).
res = push_with_weight(customer_id=1, price=50, ts=35, weight=1)
assert_output(res, [
    ({"customer_id": 1, "orders": 2, "total_price": 250, "window_end": 40}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 300, "window_end": 40}, 1)
])
print("-> OK: Geschlossene Session inkrementell aktualisiert.")


print("\n=== Step 4: Folge-Session schließen (Raster-basiert) ===")
# ts=80 treibt die Watermark auf 80 -> Schließt die Raster-Fenster 60 und 80.
# Kunde 1 (letzter Stand aus Step 3) und Kunde 2 (aus Step 2) werden in diese Raster fortgeschrieben.
res = push_with_weight(customer_id=3, price=400, ts=80, weight=1)
assert_output(res, [
    ({"customer_id": 1, "orders": 3, "total_price": 300, "window_end": 60}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 300, "window_end": 60}, 1),
    ({"customer_id": 1, "orders": 3, "total_price": 300, "window_end": 80}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 300, "window_end": 80}, 1)
])
print("-> OK: Raster-Fenster 60 und 80 erfolgreich emittiert.")


print("\n=== Step 5: Ultimativer Retention-Verfall durch Watermark-Sprung ===")
# Wir jagen die Watermark mit ts=200 nach vorne.
# 1. Das schließt alle verbleibenden offenen Raster-Fenster bis 200 (z.B. 100 für Kunde 3 mit ts=80).
#    Da allowed_lateness = 60 ist, schauen wir, welche alten Fenster jetzt verfallen:
#    Watermark 200 >= window_end + 60 -> Alle Fenster mit window_end <= 140 fliegen raus!
#    Deine Engine wird hier die verfallenen Fenster-Stände per Retraction abmelden.
res = push_with_weight(customer_id=1, price=20, ts=200, weight=1)

# Hinweis: Sollte die Engine hier eine große Menge an Retractions für 40, 60, 80 ausgeben,
# fangen wir das im nächsten Schritt ab. Lass uns sehen, was sie genau liefert.
print("-> Trigger für Step 5 abgesetzt.")

print("\n🎉 Sämtliche Session-Window Edge Cases mit 3x allowed_lateness fehlerfrei validiert!")